In [ ]:
from pathlib import Path

import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

from math import gcd

import soundfile as sf
from datasets import load_from_disk
from scipy.signal import resample_poly
from datasets import Audio
from jiwer import process_characters, process_words

In [ ]:
#load the task and model
PROJECT_ROOT = Path.cwd()

TASK = "hanzi"
# TASK = "pinyin"
# TASK = "letter"

if TASK == "hanzi":
    text_column = "hanzi"
    metric = "cer"

elif TASK == "pinyin":
    text_column = "pinyin"
    metric = "wer"

elif TASK == "letter":
    text_column = "letter"
    metric = "cer"

else:
    raise ValueError(f"Unsupported task: {TASK}")

MODEL_DIR = Path.cwd() / "model_outputs" / TASK / "finetune_v2"
if not MODEL_DIR.exists():
    raise ValueError(f"Model directory {MODEL_DIR} does not exist. Please run the training script first.")

In [ ]:
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR, local_files_only=True)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR, local_files_only=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
model.eval() # eval mode

print(f"Loaded model from {MODEL_DIR} and moved to device: {device}")

In [ ]:
DATASET_DIR = PROJECT_ROOT / "processed_data" / "aishell3_dataset"

test_dataset = load_from_disk(DATASET_DIR)["test"] # get the test split here. only the test.
target_rate = processor.feature_extractor.sampling_rate

In [ ]:
def resolve_audio_paths(batch):
    return {
        "audio": [
            str(PROJECT_ROOT / relative_path)
            for relative_path
            in batch["audio_path"]
        ]
    }

test_dataset = test_dataset.map(
    resolve_audio_paths,
    batched=True,
    desc="Resolving audio paths",
)

test_dataset = test_dataset.cast_column(
    "audio", Audio(sampling_rate=target_rate, num_channels=1)
)

# 使用一条数据进行验证


In [ ]:
example = test_dataset[0]

sample = example["audio"].get_all_samples()
waveform = sample.data.squeeze(0).numpy()

inputs = processor(
    audio=waveform,
    sampling_rate=sample.sample_rate,
    return_tensors="pt",
)

reference = example[text_column]
print(example["id"], inputs.input_values.shape, reference)

In [ ]:
inputs = inputs.to(device)

with torch.inference_mode():
    logits = model(**inputs).logits
    predicted_ids = logits.argmax(dim=-1)[0]

prediction = processor.tokenizer.decode(predicted_ids.cpu().tolist())

print("Reference :", reference)
print("Prediction:", prediction)

# 使用测试集验证

In [ ]:
results = []

with torch.inference_mode():
    for seq in test_dataset:
        sample = seq["audio"].get_all_samples()
        inputs = processor(audio = sample.data.squeese(0).numpy(),
                           sampling_rate = sample.sample_rate,
                           return_tensors = "pt").to(device)

        logits = model(**inputs).logits
        predicted_ids = logits.argmax(dim=1)[0]
        prediction = processor.tokenizer.decode(predicted_ids.cpu().tolist())

        results.append({"id": seq["id"], "reference": seq[text_column], "prediction": prediction})

In [ ]:
assert len(results) == len(test_dataset)

references = [row["reference"] for row in results]
predictions = [row["predictions"] for row in results]

if TASK == "hanzi":
    scores = process_characters(references, predictions)
    metric = "cer"
    error_rate = scores.cer

elif TASK == "letter":
    references = ["".join(text.lower().split()) for text in references]
    predictions = ["".join(text.lower().split()) for text in predictions]
    scores = process_characters(references, predictions)
    metric_name = "cer"
    error_rate = scores.cer

elif TASK == "pinyin":
    references = [" ".join(text.lower().split()) for text in references]
    predictions = [" ".join(text.lower().split()) for text in predictions]
    scores = process_words(references, predictions)
    metric_name = "wer"
    error_rate = scores.wer

else:
    raise ValueError(f"Unsupported task: {TASK}")


print(f"{TASK} test {metric_name}: {error_rate:.2%}")
print(f"Utterances: {len(results):,}")
print(f"Substitutions: {scores.substitutions:,}")
print(f"Deletions: {scores.deletions:,}")
print(f"Insertions: {scores.insertions:,}")